In [ ]:

%pip install --no-cache-dir "langchain==0.3.27" "langchain-core==0.3.79" "langchain-community==0.3.27" "langchain-text-splitters==0.3.9" "langchain-groq==0.3.8" "langchain-huggingface==0.3.1"

Note: you may need to restart the kernel to use updated packages.


In [21]:
pip uninstall -y langchain langchain-core langchain-community langchain-text-splitters langchain-groq langchain-huggingface

SyntaxError: invalid syntax (165816615.py, line 1)

In [2]:
import langchain
print(langchain.__version__)

0.3.27


In [3]:
%pip install -U sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_groq import ChatGroq

from langchain_community.document_loaders import (
    PyPDFLoader,
    DirectoryLoader
)

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

c:\Users\karth\OneDrive\Documents\vs\RAG\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\karth\AppData\Local\Temp\ipykernel_26396\2209840750.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [6]:
def file_loader(path):
    loader = DirectoryLoader(
        path,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

In [7]:
extracted_documents = file_loader(r"Data/")

In [8]:
def chunking_data(data):
    split_data =RecursiveCharacterTextSplitter (chunk_size= 500, chunk_overlap = 50)
    chunk_data = split_data.split_documents (data)
    return chunk_data

In [9]:
chunk_data = chunking_data(extracted_documents)
len(chunk_data)

125

In [10]:
chunk_data[40].page_content

'2.2  Why MLOps? — The Need \nIn 2024, as many as 88% of AI initiatives fail to reach production without a dedicated MLOps  strategy. \nML models decay as real-world data changes in ways that source code never does. MLOps addresses \nthis by providing systematic automation and governance across the ML lifecycle. \n \n1. Automates the Complete ML Lifecycle \nEliminates manual, error -prone processes across data collection, model training, testing, deployment,'

In [10]:
def get_embedding():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    return embeddings

In [11]:
embedding = get_embedding()

documents = FAISS.from_documents(
    documents=chunk_data,
    embedding=embedding
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1884.97it/s]


In [12]:
documents = FAISS.from_documents(documents=chunk_data, embedding=embedding)
documents

In [13]:
retriever = documents.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

output = retriever.invoke("What is MLOPS?")

In [14]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.6,
    api_key=GROQ_API_KEY
)

In [15]:
import langchain

print("Version:", langchain.__version__)
print("Location:", langchain.__file__)

Version: 0.3.27
Location: c:\Users\karth\OneDrive\Documents\vs\RAG\myenv\Lib\site-packages\langchain\__init__.py


In [16]:
import os
import langchain

print(os.path.exists(
    os.path.join(os.path.dirname(langchain.__file__), "chains")
))

True


In [17]:
import sys
import langchain
import langchain_core

print("Python:", sys.executable)
print("LangChain:", langchain.__version__)
print("LangChain Core:", langchain_core.__version__)

Python: c:\Users\karth\OneDrive\Documents\vs\RAG\myenv\Scripts\python.exe
LangChain: 0.3.27
LangChain Core: 0.3.79


In [19]:
import importlib.metadata as metadata

packages = [
    "langchain",
    "langchain-core",
    "langchain-community",
    "langchain-text-splitters",
    "langchain-groq",
    "langchain-huggingface",
]

for package in packages:
    try:
        print(package, "==", metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "NOT INSTALLED")

langchain == 0.3.27
langchain-core == 1.5.5
langchain-community == 0.4.2
langchain-text-splitters == 1.1.2
langchain-groq == 1.1.3
langchain-huggingface == 1.2.2


In [20]:
%pip check

langchain 0.3.27 has requirement langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.5.5.
langchain 0.3.27 has requirement langchain-text-splitters<1.0.0,>=0.3.9, but you have langchain-text-splitters 1.1.2.
Note: you may need to restart the kernel to use updated packages.


In [18]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

ModuleNotFoundError: No module named 'langchain_core.memory'

In [21]:
system_prompt = (
    "You are an expert MLOPS assistant of question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't find any related context then say that you "
    "\"don't know. Do not give any hallucinating answer of this. Use the three sentence maximum and keep the "
    "answer concise.\n\n"
    "{context}"
)

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", "{input}")
])

In [22]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.6,
    api_key=os.getenv("GROQ_API_KEY")
)

In [23]:
import importlib.metadata as md

for p in [
    "langchain",
    "langchain-core",
    "langchain-community",
    "langchain-groq",
    "langchain-huggingface"
]:
    try:
        print(p, md.version(p))
    except:
        print(p, "not installed")

langchain 0.3.27
langchain-core 0.3.79
langchain-community 0.3.27
langchain-groq 0.3.8
langchain-huggingface 0.3.1


In [27]:
import importlib.metadata as metadata

print("langchain:", metadata.version("langchain"))
print("langchain-core:", metadata.version("langchain-core"))
print("langchain-community:", metadata.version("langchain-community"))
print("langchain-groq:", metadata.version("langchain-groq"))
print("langchain-huggingface:", metadata.version("langchain-huggingface"))

langchain: 0.3.27
langchain-core: 0.3.79
langchain-community: 0.3.27
langchain-groq: 0.3.8
langchain-huggingface: 0.3.1


In [28]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("HuggingFace embeddings OK")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1782.99it/s]


HuggingFace embeddings OK


In [29]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)

response = llm.invoke("What is MLOps?")
print(response.content)

ModuleNotFoundError: No module named 'langchain_core.messages.block_translators.langchain_v0'

In [24]:
stuff_chain = create_stuff_documents_chain(llm, chat_prompt)

retriever_chain = create_retrieval_chain(retriever, stuff_chain)

question = "What is MLOPS?"

response_dict = retriever_chain.invoke({"input": question})

# response = response_dict["answer"] if isinstance(response_dict, dict) else str(response_dict)

response = response_dict["answer"]

response

ModuleNotFoundError: No module named 'langchain_core.messages.block_translators.langchain_v0'